In [1]:
from pathlib import Path
import pandas as pd
import logging

import json

from gsm_benchmarker.results_analyser.prompt_result import PromptResult
from gsm_benchmarker.results_analyser.plotting_utils import Colour
from gsm_benchmarker.results_analyser.utils import pandas_to_latex

logger = logging.getLogger('notebook')


In [2]:
MODELS = [
    'Mathstral-7B-v0.1',
     'Phi-3.5-mini-instruct',
     'phi-2',
     'gemma-2-9b',
     'Meta-Llama-3-8B-Instruct',
     'Mistral-7B-Instruct-v0.1',
     'Phi-3-mini-128k-instruct',
     'gemma-7b-it',
     'gemma-7b'
]

In [3]:
result_kwargs = dict(metric='correct', notebook=True, use_difficulty=False)
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()


pilot_gsm_result = PromptResult(
    pp / "mini_20x50x4__14_11/final",
    colour=Colour('green'),
    full_label="GSM prompt (pilot study)",
    short_label="GSM_pilot",
    **result_kwargs
)

gsm_result = PromptResult(
    pp / "default_full__06_02/final",
    colour=Colour('green'),
    full_label="GSM prompt",
    **result_kwargs
)


100%|██████████| 3/3 [00:01<00:00,  2.69it/s]


In [4]:
import os
os.path.exists('./mirzadeh-data.json')

True

In [5]:
with open('./mirzadeh-data.json') as f:
    original_results_df = pd.DataFrame(json.load(f))
    original_results_df = original_results_df.set_index('model')

original_results_df = original_results_df[original_results_df.index.isin(MODELS)]
original_results_df.columns

Index(['GSM8K_100', 'symbolic_mean', 'symbolic_variance', 'symbolic_raw'], dtype='object')

In [6]:
full_summary_df = gsm_result.mres.summary_data
full_summary_df = full_summary_df[full_summary_df.index.isin(MODELS)]
full_summary_df.columns


MultiIndex([('GSM8K',        'accuracy'),
            ('GSM8K',             'std'),
            ('GSM8K', 'strict_accuracy'),
            ('GSM8K',      'strict_std'),
            ('GSM8K',          'family'),
            ( 'main',        'accuracy'),
            ( 'main',             'std'),
            ( 'main', 'strict_accuracy'),
            ( 'main',      'strict_std'),
            ( 'main',          'family')],
           )

In [7]:
PM = "±"

In [8]:
df_combined = pd.DataFrame({
    "Mirzadeh - GSM8K": original_results_df['GSM8K_100'].apply(lambda v: f"{v:.1f}"),
    "Mirzadeh - main": original_results_df['symbolic_raw'],
    "Ours - GSM8K": full_summary_df[('GSM8K', 'accuracy')].apply(lambda v: f"{v:.1f}"),
})
df_combined.index.name = "Model"

d = {}
for row_index, row_data in full_summary_df.iterrows():
    d[row_index] = f"{row_data[('main', 'accuracy')]:.1f} ({PM} {row_data[('main', 'std')]:.2f})"
df_combined["Ours - main"] = pd.Series(d)

df_combined

,Mirzadeh - GSM8K,Mirzadeh - main,Ours - GSM8K,Ours - main
Model,,,,
Mathstral-7B-v0.1,80.0,74.0 (± 3.49),80.0,70.9 (± 3.77)
Meta-Llama-3-8B-Instruct,74.0,74.6 (± 2.94),69.0,63.4 (± 3.33)
Mistral-7B-Instruct-v0.1,42.0,30.5 (± 3.47),31.0,26.6 (± 3.38)
Phi-3-mini-128k-instruct,85.0,80.7 (± 2.94),78.0,71.7 (± 3.29)
Phi-3.5-mini-instruct,88.0,82.1 (± 3.38),81.0,71.4 (± 2.77)
gemma-2-9b,87.0,79.1 (± 2.99),71.0,57.1 (± 3.13)
gemma-7b,50.0,25.6 (± 3.25),NaN,NaN
gemma-7b-it,33.0,25.6 (± 3.25),27.0,21.1 (± 3.13)
phi-2,53.0,41.4 (± 3.56),25.0,20.0 (± 2.64)


In [9]:
print(pandas_to_latex(df_combined))

\begin{table}[t]
\begin{tabular}{lcccc}
\toprule
 & Mirzadeh - GSM8K & Mirzadeh - main & Ours - GSM8K & Ours - main \\
\midrule
Mathstral-7B-v0.1 & 80.0 & 74.0 (± 3.49) & 80.0 & 70.9 (± 3.77) \\
Meta-Llama-3-8B-Instruct & 74.0 & 74.6 (± 2.94) & 69.0 & 63.4 (± 3.33) \\
Mistral-7B-Instruct-v0.1 & 42.0 & 30.5 (± 3.47) & 31.0 & 26.6 (± 3.38) \\
Phi-3-mini-128k-instruct & 85.0 & 80.7 (± 2.94) & 78.0 & 71.7 (± 3.29) \\
Phi-3.5-mini-instruct & 88.0 & 82.1 (± 3.38) & 81.0 & 71.4 (± 2.77) \\
gemma-2-9b & 87.0 & 79.1 (± 2.99) & 71.0 & 57.1 (± 3.13) \\
gemma-7b & 50.0 & 25.6 (± 3.25) & nan & nan \\
gemma-7b-it & 33.0 & 25.6 (± 3.25) & 27.0 & 21.1 (± 3.13) \\
phi-2 & 53.0 & 41.4 (± 3.56) & 25.0 & 20.0 (± 2.64) \\
\bottomrule
\end{tabular}
\end{table}

